# 5차시 강사용 Notebook — 데이터 훑어보기와 정리(EDA)

**시연 전 안내**: Orange3로 File → Data Table → Feature Statistics를 먼저 보여준 뒤 이 노트북으로 넘어옵니다.

## 1단계. 데이터 읽고 결측값 확인하기

In [1]:
import pandas as pd

df = pd.read_csv("../../data/weekly/week05/week05_dirty_process_data.csv")
print(df.shape)
print(df.isna().sum())

(113, 9)
측정시간        0
로트번호        0
설비번호        0
공정명         0
온도_섭씨       3
압력_Pa       2
가스유량_slm    3
처리시간_sec    0
합격여부        2
dtype: int64


## 2단계. 결측값 처리하기
**설명 포인트**: 범주형(합격여부)은 제거, 숫자형은 중앙값 대체라는 원칙을 칠판에 적어둔다.

In [2]:
df = df.dropna(subset=["합격여부"])

for col in ["온도_섭씨", "압력_Pa", "가스유량_slm"]:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

print(df.isna().sum().sum())

0


## 3단계. 중복값 찾고 제거하기

In [3]:
print(df.duplicated().sum())
df = df.drop_duplicates()
print(df.shape)

3
(108, 9)


## 4단계. 이상값 찾기
**어려워할 부분**: isna()로 이상값이 안 잡힌다는 것을 직접 실행해서 보여준다.

In [4]:
hot_outlier = df[df["온도_섭씨"] > 400]
neg_pressure = df[df["압력_Pa"] < 0]
bad_gas = df[(df["가스유량_slm"] < 0) | (df["가스유량_slm"] >= 200)]

print(len(hot_outlier), len(neg_pressure), len(bad_gas))

2 2 2


## 5단계. 이상값 제거하기

In [5]:
df = df[df["온도_섭씨"] <= 400]
df = df[df["압력_Pa"] >= 0]
df = df[(df["가스유량_slm"] >= 0) & (df["가스유량_slm"] < 200)]

print(df.shape)

(102, 9)


## 6단계(종합). 정제 전후 평균 비교하기

In [6]:
after_mean = df["온도_섭씨"].mean()
print(f"정제 전 평균 온도: 304.9도 (113행)")
print(f"정제 후 평균 온도: {after_mean:.1f}도 ({len(df)}행)")

정제 전 평균 온도: 304.9도 (113행)
정제 후 평균 온도: 300.6도 (102행)


## 7단계. 오늘의 학습을 한 문장으로 정리하기
**진행 방법**: 6단계에서 본 정제 전후 평균 숫자를 직접 인용해 말하게 한다. 숫자가 들어간 문장이 훨씬 설득력 있다는 것을 이 자리에서 짚어준다.
**설명 포인트**: 결과 해석 문장 쓰기는 1차시부터 10차시까지 매 차시 빠지지 않는 고정 활동이다(docs/curriculum.md 「차시 간 연결 원칙」). 코드를 친 것으로 끝내지 않고 자기 말로 바꿔 말해보게 하는 것이 이 과정의 목표 — "데이터로 공정 상태를 설명할 수 있는 사람" — 에 직접 닿는 활동이므로 시간이 모자라도 2~3분은 반드시 확보한다.

> (예시 답안) 결측값은 제거하거나 대체하고, 중복값은 drop_duplicates()로, 이상값은 조건 검색으로 찾아 제거한다. 이상값 하나가 평균 같은 요약값을 크게 왜곡할 수 있다.

## 오류 대처 방법
- `KeyError` 발생 시: `df.columns`로 정확한 열 이름 확인.
- fillna 대상 지정 실수: 반드시 `df["열"] = df["열"].fillna(...)` 형태인지 확인.

## 확장 실습(빠른 학습자용)
공정명 오타를 str.strip()/replace()로 통일해보게 한다.

In [7]:
df["공정명"] = df["공정명"].str.strip().replace({"중착": "증착", "포토공정": "포토"})
print(df["공정명"].unique())

<ArrowStringArray>
['포토', '산화', '세정', '증착', '식각']
Length: 5, dtype: str


## 8단계. AI에게 질문하며 더 알아보기
**설명 포인트**: IQR 질문이 나오면, 오늘 배운 "정해진 정상 범위(295~305도)로 찾기"와 "통계적으로
자동 계산하기(IQR)"의 차이를 비교해서 설명해준다.

질문 예시:
- "결측값을 평균으로 채우는 것과 중앙값으로 채우는 것은 언제 다르게 써야 하나요?"
- "이상값(outlier)을 찾는 통계적 방법(IQR, Z-score)에는 어떤 것들이 있나요?"
- "drop_duplicates()에서 특정 열 기준으로만 중복을 판단하려면 어떻게 하나요?"
- "결측값을 무조건 채우거나 지우는 것 말고 다른 처리 방법도 있나요?"

## 9단계. AI에게 코드 생성 요청하고 직접 실행해보기
**설명 포인트**: `quantile()`/IQR은 오늘 배우지 않은 통계 개념이다 — "정상 범위를 사람이 정하는
방식"과 "통계로 자동 계산하는 방식"을 비교해서 설명해준다.

프롬프트 예시: "숫자 데이터에서 IQR(사분위범위) 방법으로 이상값을 찾아주는 코드를 만들어줘."

아래는 AI가 생성해줄 수 있는 예시 코드와 실행 결과다.

In [8]:
temps = pd.Series([295.4, 298.1, 312.6, 301.0, 296.8, 850.0])

q1, q3 = temps.quantile(0.25), temps.quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

outliers = temps[(temps < lower) | (temps > upper)]
print(outliers)

5    850.0
dtype: float64

## 10단계. 연습문제 — 나만의 정제 리포트 만들기
**설명 포인트**: 1~3단계에서 배운 `isna()`·`duplicated()` 스킬을 하나의 "정제 리포트" 문장으로
요약하는 연습이다. `df`는 이미 여러 단계를 거치며 정제되었으므로 원본과 비교하려면 `raw_df`로
파일을 다시 읽어야 한다는 점을 짚어준다 — 실무에서도 이런 "정제 보고서"를 남기는 습관이 중요하다.

In [9]:
raw_df = pd.read_csv("../../data/weekly/week05/week05_dirty_process_data.csv")
missing_count = raw_df["합격여부"].isna().sum()
duplicate_count = raw_df.duplicated().sum()

print(f"정제 리포트: 결측 {missing_count}건, 중복 {duplicate_count}건을 확인했습니다.")
print(f"정제 후 최종 데이터는 {len(df)}행입니다.")

정제 리포트: 결측 2건, 중복 3건을 확인했습니다.
정제 후 최종 데이터는 102행입니다.


## 11단계. AI로 재미있는 미니 프로그램 만들기 🎉 — 이상값 탐정 게임
**설명 포인트**: 오늘 배운 "정상 범위를 벗어난 값 찾기"를 통계(평균 ± 표준편차)로 재미있게
재구성한 것이다. `statistics` 모듈은 아직 배우지 않았지만, 온도_섭씨 이상값을 찾던 방식(조건
검색)과 원리가 같다는 점을 연결해서 설명하면 좋다.

프롬프트 예시: "숫자 리스트를 받아서, 평균에서 표준편차의 2배 이상 벗어난 값을 '용의자'로 찾아
🔍 이모지와 함께 출력하는 파이썬 코드를 만들어줘."

아래는 AI가 생성해줄 수 있는 예시 코드다.

In [10]:
import statistics

sample_values = [301.2, 298.5, 305.0, 612.3, 299.9, 300.1, 15.0, 302.4]

mean_v = statistics.mean(sample_values)
std_v = statistics.stdev(sample_values)

print("🕵️ 이상값 탐정 게임 시작!")
for v in sample_values:
    if abs(v - mean_v) > 2 * std_v:
        print(f"🔍 용의자 발견! {v} (평균에서 많이 벗어났어요)")
    else:
        print(f"{v} → 정상 범위")

🕵️ 이상값 탐정 게임 시작!
301.2 → 정상 범위
298.5 → 정상 범위
305.0 → 정상 범위
612.3 → 정상 범위
299.9 → 정상 범위
300.1 → 정상 범위
15.0 → 정상 범위
302.4 → 정상 범위
